# E9 GNN Navigation

Author: Arush Arora

## Introduction

This codebase has mostly consisted of additive Graph Positional Encodings (GREPs) injections to provide nodal embeddings to the LLMs at hand. This new multi-stage training will rely on training an R-PEARL/GT to simply replicate the shortest-distance paths of the graph before expecting it to serve the LLM with **multiplicative** GREPs for navigation tasks, which will be factored directly into the attention-mask matrix for rendition to the LLM (as a Hadamard product cover on the attention logits). Thus, the system will be more carefully trained to incorporate the variation in model architecture among GNNs and LLMs (in terms of their pre-trained weights rather than simply their mathematical foundations).

## Mathematical Overview

### The R-PEARL GNN

The Random Positional Encoding (R-PEARL) GNN architecture is a PE generator that inputs white noise and processes it over an undirected graph $\mathcal{G} = (\mathcal{V}, \mathcal{E}, \mathcal{W})$. In this work, the graph is represented by an adjacency matrix $A$, and the GNN composes [Topology Adaptive Graph (TAG)](https://arxiv.org/abs/1710.10370) Convolutional Layers with pointwise nonlinearities (demodulators).

#### Graph Convolutional Network (GNN)

The code below establishes this project's implementation of a Graph Convolutional Network, which is the foundational architecture comprising R-PEARL. The equation to demonstrate the internal architecture of this NN as follows (in most cases, $\mathbf{P}(\cdot) = \mathbf{I}(\cdot)$, where $\mathbf{I}$ is the identity function):
$$\Phi(\mathbf{X}, \mathbf{S}, \mathcal{H}) = \mathbf{X}^{(L)}$$
$$\mathbf{X}^{(0)} = \mathbf{X} \qquad \mathbf{X}^{(l)} = \mathbf{P}\Bigg[\sigma\Bigg(\sum_{k = 0}^{K^{(l)} - 1} \mathbf{S}^k\mathbf{X}^{(l - 1)}\mathbf{H}_k^{(l)}\Bigg)\Bigg]$$

#### Random Graph Positional Encodings (R-PEARL)

The R-PEARL architecture extends on the GCN by instantiating it with simply one layer – a TAG Convolution and Demodulator. The mathematical equations below express the functionality of the R-PEARL network:
1. The white-noise matrix is sampled from the Gaussian distribution. $$\mathbf{Q} \in \mathbb{R}^{M \times N} \qquad \mathbf{Q} \sim \mathcal{N}(0, \mathbf{I}) \qquad \mathbf{Q} = \begin{bmatrix}
  \mathbf{q}^{(0)} & \cdots & \mathbf{q}^{(m)} & \cdots & \mathbf{q}^{(M)}
  \end{bmatrix}$$

2. The R-PEARL network has row-vector parameter $\mathbf{H}^{(0)} \in \mathbb{R}^{1 \times D}$. It takes in each column of the white-noise matrix individually and produces a sample $\mathbf{P}^{(m)} \in \mathbb{R}^{N \times D}$, which are then pooled to form GREP $\mathbf{P}$:$$\mathbf{P}^{(m)} = \Phi\Big(\mathbf{q}^{(m)};\, \mathbf{S}, \mathcal{H}\Big) = \sigma\bigg(\sum_{k = 0}^{K = 1} \mathbf{S}^k\mathbf{q}^{(m)} {\mathbf{H}}_k\bigg)$$ $$\mathbf{P} = \mathbb{\hat{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \mathbf{P}^{(m)}$$

### Sparse Graph Transformer

The Sparse Graph Transformer (hereafter named Graph Transformer or GT) follows the same architecture as that of a normal transformer, albeit that the attention mecahnism is modified to scope only over the $k$-hop neighborhood of the query node. The mathematical equations below express the functionality of the Graph Transformer:
$$\mathbf{X}_L = \Phi\Big(\mathbf{X}_0 + \mathbb{\hat{E}}_{\mathbf{q \sim \mathcal{N}(0,\, \mathbf{I})}}\big[\Phi(\mathbf{q};\, S, \mathcal{H})\big];\, \mathcal{T}\Big)$$
$$\mathbf{A}^{(h)}_l = \left[\frac{\exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1,\ U}\right)\right]}{\mathbf{1}^\top \exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1,\ U}\right)\right]}\right]^\top_{\begin{subarray}{l}t \in [N] \\[2.5pt] U = \mathcal{N}^{\le k}(t)\end{subarray}}$$
$$\mathbf{Y}^{(h)}_l = \left(\mathbf{W}_o\right)^\top_l \mathbf{V}_l \mathbf{X}_{l - 1} \left(\mathbf{A}^{(h)}_l\right)^\top$$
$$\mathbf{X}_l = \sigma\bigg(\sum_{h = 1}^H \mathbf{Y}^{(h)}_l\bigg)$$

### Transformer

The Transformer architecture follows that of the Llama3.1-8B distilled PRISM model. First, the TXT file, containing the scene-graph data, is tokenized and embedded into matrices $E$ and $\tilde{X}$ as follows, where $V$ is the size of the vocabulary and $d$ is the embedding dimension.

$$\text{TXT Tokenized Data from GPT-4: } E = \begin{bmatrix}
\mathbf{e}_1 & \mathbf{e}_2 & \overset{\mathbf{e}_t}{\cdots} & \mathbf{e}_T
\end{bmatrix}^\top \qquad \mathbf{e}_t \in \mathbb{R}^V$$

$$\text{Embed: } X = \begin{bmatrix}
\mathbf{x}_1 & \mathbf{x}_2 & \overset{\mathbf{x}_t}{\cdots} & \mathbf{x}_T
\end{bmatrix}^\top \qquad \mathbf{x}_t \in \mathbb{R}^d$$

Next, the transformer operates using the equations below:

$$\mathbf{X} = \mathbf{\tilde{X}} + \mathbf{P}$$

$$\mathbf{Z}_{1:t}^{(L)} = \operatorname{Trf}\Big(\mathbf{X}_{1:t}, {\mathcal{T}}_l\Big) \qquad {\mathcal{T}}_l = \begin{bmatrix}
\mathbf{Q}_l & \mathbf{K}_l & \mathbf{V}_l & \left(\mathbf{W}_o\right)_l
\end{bmatrix}^\top \in \mathbb{R}^{4 \times T \times D}$$

$$\hat{\mathbf{Y}}_{t + 1} = \operatorname{Linear}\Big(\mathbf{Z}_{1:t}^{(L)}\Big) \in \mathbb{R}^V$$
$$\text{Cross-Entropy Loss: } \mathcal{L}(E, \hat{\mathbf{Y}}) = \sum_t \sum_v e_{vt}\log{\hat{y}_t}$$

### Graph-Augmented LLM

The last class that is needed to create the full GREP-PRISM architecture is the `GraphAugmentedLLM`, which simply implements the following equation as a Neural Network object in PyTorch's `torch.nn` module (referring to above equations for definitions).
$$\mathbf{P} = \mathbb{\hat{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \Phi\Big(\mathbf{q}^{(m)}, \mathbf{S}, \mathcal{H}\Big)$$

$$\mathbf{X} = \mathbf{\tilde{X}} + \mathbf{P}$$

$${\mathbf{Z}}_{1:t}^{(L)} = \operatorname{Trf}\Big({\mathbf{X}}_{1:t}; \, \cdot \,\Big)$$

## Setup

In [1]:
# Import modules.
import copy
import torch
import random
import wandb
import sympy as sp
import networkx as nx

from torch import nn
from torch_geometric.data import Data
from torch.nn.utils import clip_grad_norm_
from itertools import product, combinations
from torch_geometric.utils import to_networkx
from torch.optim.lr_scheduler import ReduceLROnPlateau

from prism.models.r_pearl import RandomGNNPositionalEncodings
from prism.models.gt import GraphTransformer
from prism.models.gcn import GCN
from prism.eval import evaluate
from prism.data import data

In [2]:
# Weights & Biases setup. Mirrors prism.training.train_v3._setup_wandb (project /
# name / tags / group + full-config logging), adapted for this notebook's hand-written
# train loops. Each training stage gets its own run, grouped/tagged by GNN type so the
# R-PEARL and GT variants of the same stage line up on one W&B dashboard. The helpers
# introspect the live optimizer / scheduler / loss objects so EVERY hyperparameter is
# logged without hand-maintaining a list.
WANDB_PROJECT = 'e9-gnn-navigation'


def optimizer_hparams(optimizer):
    """Every optimizer setting: class name, shared defaults, and per-param-group values
    (LRs, betas, eps, weight_decay, ...) with the parameter tensors stripped out."""
    return {
        'optimizer': type(optimizer).__name__,
        'optimizer_defaults': dict(optimizer.defaults),
        'param_groups': [
            {k: v for k, v in g.items() if k != 'params'}
            for g in optimizer.param_groups
        ],
    }


def scheduler_hparams(scheduler):
    """Every LR-scheduler setting (or {'scheduler': None} when unused)."""
    if scheduler is None:
        return {'scheduler': None}
    keys = ('mode', 'factor', 'patience', 'threshold', 'threshold_mode',
            'cooldown', 'min_lrs', 'eps')
    return {
        'scheduler': type(scheduler).__name__,
        **{k: getattr(scheduler, k) for k in keys if hasattr(scheduler, k)},
    }


def loss_hparams(loss_fn):
    """Loss class, reduction, and pos_weight (resolved to plain Python)."""
    out = {'loss_fn': type(loss_fn).__name__,
           'reduction': getattr(loss_fn, 'reduction', None)}
    pos_weight = getattr(loss_fn, 'pos_weight', None)
    if pos_weight is not None:
        out['pos_weight'] = (pos_weight.detach().cpu().tolist()
                             if torch.is_tensor(pos_weight) else pos_weight)
    return out


def init_wandb(stage, hparams):
    """Start a W&B run for a training `stage` ('edge_detection' / 'path_navigation').

    Logs the FULL run config: the GNN construction kwargs (`model_hparams`, set in the
    GNN-instantiation cell) plus every optimizer / scheduler / loss / batching
    hyperparameter the caller assembles in `hparams`. `model_type` selects R-PEARL vs
    GT and drives the run name / tag / group. Returns the run; `reinit=True` so
    successive stages in one notebook session each open a fresh run.
    """
    return wandb.init(
        project=WANDB_PROJECT,
        name=f'{stage}_{model_type}',
        tags=[stage, model_type],
        group=model_type,
        config={'model_type': model_type, 'stage': stage,
                'model': model_hparams, **hparams},
        reinit='return_previous',
    )

In [3]:
# Define a tensor rendering function.
def render_matrix(mat: torch.tensor, sig_figs: int = 3, decimals: int = 0):
    out = sp.Matrix(mat.detach().cpu().numpy())
    if sig_figs > 0:
        return sp.N(out, sig_figs)
    if decimals > 0:
        return out.applyfunc(lambda x: x.round(decimals))
    return out

In [4]:
# Standard options.
eval_path = '../data/revised/gen/nav100_n30_gemma_data/split/test_graphs'
device = 'cuda'

In [5]:
# Setup eval infrastructure.
samples_by_graph, graph_file_by_name = data.load_samples_by_graph(eval_path)
graph_file = random.choice(list(samples_by_graph.keys()))
eval_data = samples_by_graph[graph_file]
eval_data = {graph_file: [random.choice(eval_data)]}

In [6]:
print(f'/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/{graph_file}.html')

/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/data_gen_045.html


## Experiments

### §1 Pretraining a GNN to Classify Edge Existence

We first hope to optimize a GNN (R-PEARL or Graph Transformer) to classify whether an edge exists in the graph or not. Such a model will serve as a backbone pretrained model for fine-tuning on reporting shortest paths. The equations to represent this procedure are below:
$$\mathbf{H} = \Phi\Big(\mathbb{\hat{E}}_{\mathbf{q}}\big[\Phi(\mathbf{q};\, S, \mathcal{H}\,)\big];\, \mathcal{T}\Big)$$
$$\mathbf{\hat{y}}_{ij} = \text{MLP}\big[\mathbf{h}_i\ \Vert\ \mathbf{h}_j\ \Vert\ \mathbf{h}_i \odot \mathbf{h}_j\ \Vert\ |\mathbf{h}_i - \mathbf{h}_j\|\big] \in [0, 1]$$

#### Model Definitions

We first define the models.

In [7]:
# Instantiate a GNN. `model_type` / `model_hparams` are exposed at module scope so
# init_wandb can log the GNN config; create_gnn writes model_hparams as it builds.
def create_gnn(model_type: str):
    global model_hparams
    if model_type == 'gt':
        model_hparams = dict(
            num_layers=3,
            pe_hidden_channels=256,
            pe_num_layers=5,
            d_model=1024,
            heads=8,
            num_samples=320,
            dropout=0.1,
            k_pe=3,
            k_gt=2,
            eps=1e-6,
            use_layer_norm=True,
        )
        gnn = GraphTransformer(**model_hparams)
        gnn.out_features = gnn.d_model
    else:
        model_hparams = dict(
            pe_hidden_channels=256,
            pe_num_layers=5,
            d_model=1024,
            num_samples=320,
            dropout=0.1,
            k=3,
            eps=1e-6,
            use_layer_norm=True,
        )
        gnn = RandomGNNPositionalEncodings(**model_hparams)
        gnn.out_features = gnn.output_projection.out_features
    return gnn


model_type = 'gt'
gnn = create_gnn(model_type)

In [8]:
from typing import Union


# Define a class for edge detection and instantiate it.
class GNNEdgeDetector(nn.Module):
    """
    Simple class to detect whether Node 1 and Node 2 are connected by
    applying an MLP to the Graph Positional Encodings of both nodes concatenated.
    """
    def __init__(self, gnn: Union[RandomGNNPositionalEncodings, GraphTransformer]):
        super(GNNEdgeDetector, self).__init__()
        self.gnn = gnn
        shape = self.gnn.out_features
        self.classifier = nn.Sequential(
            nn.Linear(4 * shape, shape),
            nn.LeakyReLU(),
            nn.Linear(shape, 1)
        )
        self.graph = Data(
            x=torch.empty((0, 0), dtype=torch.float),
            edge_index=torch.empty((2, 0), dtype=torch.long)
        )
        self.cached_pe = torch.zeros(size=(1, shape))

    def forward(self, graph: Data, node1: int, node2: int):
        if self.cached_pe is None or not self.cached_pe.any() or self.graph is not graph:
            self.graph = graph
            self.cached_pe = self.gnn(self.graph)
        hi, hj = self.cached_pe[node1], self.cached_pe[node2]
        return self.classifier(torch.cat((hi, hj, hi * hj, abs(hi - hj)), dim=0))
    
    def invalidate_cache(self):
        self.graph = None
        self.cached_pe = None


# Instantiate the class.
detector = GNNEdgeDetector(gnn)

#### Numeric Visualizations with SymPy

Using the `render_matrix()` function defined at the very beginning of this notebook, we explore the procedure needed to preprocess a pre-training set for the GNN to reconstruct the graph adjacency given a scene graph PyTorch `Data` object.

In [9]:
# Prepare a graph from the data to be used in the GNN.
from torch_geometric.utils import to_dense_adj, to_networkx
from prism.data import utils
import numpy as np
import sympy as sp

# Prepare the graph for rendition.
ex_graph = utils.scene_graph_dict_to_pyg(eval_data[graph_file][0][2])
adj = to_dense_adj(ex_graph.edge_index).squeeze().cuda()
ex_graph.edge_index = ex_graph.edge_index.to(device)
ex_graph.x = ex_graph.x.to(device)

# Show the adjacency matrix of the graph.
render_matrix(adj)

Matrix([
[  0,   0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0],
[  0,   0,   0,   0,   0,   0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0,   0,   0, 1.0, 1.0,   0, 1.0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0, 1.0,   0, 1.0,   0,   0,   0, 1.0, 1.0,   0,   0,   0,   0, 1.0,   0,   0,   0, 1.0

In [10]:
# Feed the matrix to the GNN.
out = gnn(ex_graph).to(device)
_, _, V = torch.pca_lowrank(out, q=10, center=True)
out = out - out.mean(dim=0)
render_matrix(out @ V)

Matrix([
[-4.76,   1.21,   1.97,    -1.16,  -0.47,  -0.209,   -3.43,    2.62,   -1.03,   -2.05],
[-4.88,   0.85,  0.695,     2.06,  -2.56,   0.976,   -3.65,   0.125,    1.79,    2.68],
[ 4.67, -0.456, -0.417,   -0.209,  -3.54,   -4.79,   0.658,   -1.52,   0.781,   -1.51],
[ 1.78,  -1.88,  0.622,     1.24,   1.12,    2.37,   0.287,   -1.18,    1.05,    1.93],
[ 2.05,  -1.33, -0.241,     1.92,  -1.35,    -1.9,   0.405,    4.32,    1.86,     1.5],
[-3.92,   1.96,   2.53,     5.02, 0.0217,  -0.439,    3.94,   0.406,  -0.331,   0.951],
[-3.98,   1.31,  -1.93, -0.00399,   -3.5,   0.737,    1.01,   -4.52,    2.69,   -2.34],
[-3.91, -0.446,   2.94,  -0.0211,  -1.01,   0.251,    3.18,    1.61,  -0.536,   -1.82],
[-4.94,   1.34, -0.971,   -0.596,  0.502,   0.483, -0.0121,   0.672,   0.642,     2.0],
[-4.76,  -2.42,  -3.57,    -1.69,    2.6, -0.0893,   0.344,    1.42,   0.545,   0.268],
[-4.06, -0.846,  0.921,    0.134,  0.309,   -1.98,   0.542,   -1.72,   -1.77,   -1.56],
[-5.07,  0.712,  -3.19,

In [11]:
# Test out the Detector.
node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
out = detector(ex_graph, node1, node2).to(device)
render_matrix(torch.sigmoid(out))

Matrix([[0.544]])

#### Pre-Training of GNN on Edge Incidence

Next, we actually preprocess and train the GNN using the steps defined above.

In [12]:
# Import Modules.
from torch_geometric.loader import DataLoader

# Configure the training and test datasets.
train_dataset, _ = data.load_samples_by_graph(
    '../data/revised/gen/nav100_n30_gemma_data/split/train_graphs'
)
test_dataset, _ = data.load_samples_by_graph(
    '../data/revised/gen/nav100_n30_gemma_data/split/test_graphs'
)

# Configure the validation dataset.
train_prop = 0.8
train_num = len(train_dataset)
train_keys = random.sample(list(train_dataset.keys()), k=int(train_num * train_prop))
val_dataset = {k: v for k, v in train_dataset.items() if k not in train_keys}
train_dataset = {k: v for k, v in train_dataset.items() if k in train_keys}

# Add edge existence tuples for edge existence.
def generate_data_edges(dataset):
    graphs = [utils.scene_graph_dict_to_pyg(v[0][2]) for _, v in dataset.items()]
    for graph in graphs:
        graph.x = graph.x.to(device)
        graph.edge_index = graph.edge_index.to(device)
        combs = torch.tensor(
            [[u, v] for u in range(graph.num_nodes) for v in range(u + 1, graph.num_nodes)],
            device=device
        ).T
        existence = torch.tensor(
            [combs[:, i].tolist() in graph.edge_index.T.tolist() for i in range(combs.shape[1])],
            device=device
        )
        graph.exclusion = combs[:, ~existence].to(device)
        indices = torch.randint(
            high=graph.exclusion.shape[1], size=(graph.edge_index.shape[1],), device=device
        )
        graph.edges_x = torch.cat((graph.edge_index, graph.exclusion[:, indices]), dim=1).to(device)
        graph.edges_y = torch.cat(
            (torch.ones((graph.edge_index.shape[1],)), torch.zeros((indices.shape[0],))), 
            dim=0
        ).to(device)
    
    return graphs


def reshuffle(graphs):
    for graph in graphs:
        indices = torch.randint(
            high=graph.exclusion.shape[1], size=(graph.edge_index.shape[1],), device=device
        )
        graph.edges_x = torch.cat((graph.edge_index, graph.exclusion[:, indices]), dim=1).to(device)
        graph.edges_y = torch.cat(
            (torch.ones((graph.edge_index.shape[1],)), torch.zeros((indices.shape[0],))), 
            dim=0
        ).to(device)
    
    return graphs


train_graphs_edges = generate_data_edges(train_dataset)
val_graphs_edges = generate_data_edges(val_dataset)
test_graphs_edges = generate_data_edges(test_dataset)

In [13]:
# Train the GNNEdgeDetector to reconstruct the graph adjacency.
batch_size = 4
val_freq = 5
epochs = 150
es_patience = 5

def test_loop_edges(dataloader, model, loss_fn, wandb_prefix=None, epoch=None):
    model.to(device)
    model.eval()
    size = len(dataloader.dataset)
    test_loss, correct = 0, 0
    tp = fp = fn = tn = 0

    with torch.no_grad():
        for graph in dataloader.dataset:
            preds = torch.stack([
                model(graph, graph.edges_x[0, k], graph.edges_x[1, k])
                for k in range(graph.edges_x.shape[1])
            ]).squeeze(-1).to(device)
            test_loss += loss_fn(preds, graph.edges_y).item()
            true = graph.edges_y.bool()
            pred = preds > 0
            tp += (pred & true).sum().item()
            fp += (pred & ~true).sum().item()
            fn += (~pred & true).sum().item()
            tn += (~pred & ~true).sum().item()
            correct += ((preds.sigmoid() > 0.5).float() == graph.edges_y).float().mean().item()

    test_loss /= size
    correct /= size
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    bal_acc = 0.5 * (recall + tn / (tn + fp + 1e-9))
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, F1: {f1:.3f} | P: {precision:.3f} "
          f"| R: {recall:.3f} | Bal Acc: {(100*bal_acc):.1f}% | Avg loss: {test_loss:>8f} \n")
    # Log eval metrics under the given split prefix (e.g. 'val') when a run is active.
    if wandb_prefix is not None and wandb.run is not None:
        log = {
            f'{wandb_prefix}/loss': test_loss,
            f'{wandb_prefix}/accuracy': correct,
            f'{wandb_prefix}/f1': f1,
            f'{wandb_prefix}/precision': precision,
            f'{wandb_prefix}/recall': recall,
            f'{wandb_prefix}/bal_acc': bal_acc,
        }
        if epoch is not None:
            log['epoch'] = epoch
        wandb.log(log)
        # Final test metrics: also surface as run-summary headline numbers.
        if wandb_prefix == 'test':
            wandb.run.summary.update({k: v for k, v in log.items() if k != 'epoch'})
    return test_loss


def train_loop_edges(train_dataloader, val_dataloader, test_dataloader, model,
               loss_fn, optimizer, scheduler, batch_size=20, epochs=50):
    size = len(train_dataloader.dataset)
    model.to(device)
    model.train()
    val_loss: float = 0
    best_val, best_state, bad_runs = float('inf'), None, 0
    # Log the full run config: GNN (model_hparams) + optimizer/scheduler/loss/batching.
    run = init_wandb('edge_detection', {
        'batch_size': batch_size, 'epochs': epochs,
        'val_freq': val_freq, 'es_patience': es_patience,
        **optimizer_hparams(optimizer),
        **scheduler_hparams(scheduler),
        **loss_hparams(loss_fn),
    })
    global_step = 0
    for i in range(epochs):
        # Validation loop.
        if i % val_freq == 0:
            print(f"=============\nValidation #{i // val_freq}\n=============")
            val_loss = test_loop_edges(val_dataloader, model, loss_fn, wandb_prefix='val', epoch=i)
            if scheduler:
                scheduler.step(val_loss)
            if val_loss < best_val - 1e-3:
                best_val, bad_runs = val_loss, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                bad_runs += 1
                if bad_runs >= es_patience:
                    print(f"Early stop at epoch {i} (best val {best_val:>8f})")
                    break
            model.train()
        
        print(f"=============\nEpoch #{i}\n=============")
        optimizer.zero_grad()
        reshuffle(train_dataloader.dataset)
        for j, graph in enumerate(train_dataloader.dataset):
            # Compute prediction and loss.
            preds = torch.stack([
                model(graph, graph.edges_x[0, k], graph.edges_x[1, k])
                for k in range(graph.edges_x.shape[1])
            ]).squeeze(-1).to(device)
            loss = loss_fn(preds, graph.edges_y)

            # Backpropagation.
            (loss / batch_size).backward()

            # Optimization and results.
            if (j + 1) % batch_size == 0:
                clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

                global_step += 1
                loss, current = loss.item(), j
                wandb.log({
                    'train/loss': loss,
                    'train/lr': optimizer.param_groups[0]['lr'],
                    'epoch': i,
                    'global_step': global_step,
                })
                print(f"Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
        
    # Early stopping hatch.
    if best_state is not None:
        model.load_state_dict(best_state)
    
    # Test the finished model.
    test_loop_edges(test_dataloader, model, loss_fn, wandb_prefix='test')
    run.finish()


loss_fn = nn.BCEWithLogitsLoss()
train_dataloader = DataLoader(train_graphs_edges, batch_size=batch_size)
val_dataloader = DataLoader(val_graphs_edges, batch_size=batch_size)
test_dataloader = DataLoader(test_graphs_edges, batch_size=batch_size)
optimizer = torch.optim.AdamW([
    {'params': detector.gnn.parameters(), 'lr': 3e-5},
    {'params': detector.classifier.parameters(), 'lr': 3e-4},
], betas=(0.9, 0.95), weight_decay=0.05)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
# train_loop(train_dataloader, val_dataloader, test_dataloader, detector,
#            loss_fn, optimizer, scheduler, batch_size=batch_size, epochs=epochs)

In [14]:
# torch.save(detector, '../outputs/e9_multistage_training/edge_detector.pt')
# torch.save(detector.gnn.state_dict(), f'../outputs/e9_multistage_training/edge_detector_{model_type}.pt')
detector = torch.load('../outputs/e9_multistage_training/edge_detector_final.pt', weights_only=False)
gnn.load_state_dict(torch.load(f'../outputs/e9_multistage_training/edge_detector_{model_type}_final.pt'))

<All keys matched successfully>

#### Evaluation of Pre-Trained GNN on Edge Incidence

We now test the trained model on the evaluation dataset. First, we we will render the output for clarity.

In [15]:
# Test out the Detector.
detector.to(device)
node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
out = detector(ex_graph, node1, node2)
render_matrix(torch.sigmoid(out))

Matrix([[4.61e-17]])

In [16]:
# Evaluate the GNN on its reconstruction of test graph adjacencies.
test_loop_edges(test_dataloader, detector, loss_fn)

Test Error: 
 Accuracy: 91.8%, F1: 0.923 | P: 0.860 | R: 0.996 | Bal Acc: 91.7% | Avg loss: 0.247562 



0.24756246209144592

### §2 Fine-tuning the GNN to Classify Shortest-Path Node Inclusion

We now wish to optimize the pre-trained GNN (R-PEARL or Graph Transformer) to classify which nodes reside on the shortest path between two given nodes in the graph. Such a model will serve as the backbone for the multi-stage training loop featured in E9 Multistage Training of the GREP-PRISM project. The equations to represent this procedure are below:
$$\mathbf{H} = \Phi\Big(\mathbb{\hat{E}}_{\mathbf{q}}\big[\Phi(\mathbf{q};\, S, \mathcal{H}\,)\big];\, \mathcal{T}\Big)$$
$$\mathbf{H} = \mathbf{\Psi} \implies \overline{\mathbf{\Psi}\mathbf{\Psi}}^\top = C \approx [SPD]$$
$$\overline{\mathbf{\Psi}\mathbf{\Psi}}^\top = (\mathbf{\Psi} - \mathbf{\overline{\Psi}})(\mathbf{\Psi} - \mathbf{\overline{\Psi}})^\top, \mathbf{\overline{\Psi}} = \left[\frac{\underbar{e}_i^\top \mathbf{\Psi}}{\Vert \underbar{e}_i^\top \mathbf{\Psi} \Vert_2}\right]_{i \in [N]}$$
$$\mathbf{E} = \mathbb{E}\left[\frac{[SPD]_{ij}}{\delta(i, j)}\right]_{i, j \in [N]}$$

#### Model Definitions

We first define the model by attaching a simple GCN head to the GNN positional encoder.

In [17]:
# Define a class for shortest-path distance estimation and instantiate it.
class GNNShortestPathsEstimator(nn.Module):
    """
    Simple class to predict the shortest-path distance graphical lasso estimator (covariance).
    """
    def __init__(self, gnn: Union[RandomGNNPositionalEncodings, GraphTransformer]):
        super(GNNShortestPathsEstimator, self).__init__()
        self.head = GCN(
            model_hparams['d_model'],
            model_hparams['d_model'],
            model_hparams['num_layers'],
            skip_connection=True,
            dropout=model_hparams['dropout'],
            k=model_hparams['k_pe']
        )
        self.gate = nn.Parameter(torch.tensor(0.1))
        self.gnn = gnn

    def forward(self, graph: Data):
        graph = graph.clone()
        graph.x = self.gnn(graph)
        out = self.head(graph)
        out = torch.cdist(out, out, p=2)
        return self.gate * out

#### Numeric Visualizations with SymPy

Using the `render_matrix()` function defined at the very beginning of this notebook, we first explore the procedure needed to preprocess a pre-training set for the GNN to reconstruct the shortest-path distances matrix given a scene graph PyTorch `Data` object.

In [18]:
# Prepare example graph.
EPS=1e-12
N = ex_graph.num_nodes
ex_graph.paths = torch.zeros((N, N, N)).to(device)
ex_graph.dist = torch.full((N, N), float('inf')).to(device)
g = to_networkx(ex_graph, to_undirected=True, edge_attrs=['distance_m'])
lengths = dict(nx.all_pairs_dijkstra_path_length(g, weight='distance_m'))
for u in range(N):
    for v in lengths[u]:
        ex_graph.dist[u, v] = lengths[u][v]
        for p in nx.all_shortest_paths(g, u, v, weight='distance_m'):
            ex_graph.paths[u, v, p] = 1
            ex_graph.paths[v, u, p] = 1
ex_graph.dist.fill_diagonal_(EPS)
pass

In [19]:
# Test out the SPD GNN.
spd_gnn = GNNShortestPathsEstimator(gnn)
out = spd_gnn(ex_graph).to(device)
render_matrix(out)

Matrix([
[  0.141,   308.0,   240.0,   194.0,   162.0, 2.05e+3, 1.42e+3, 1.01e+3, 1.73e+3,   850.0, 1.25e+3, 2.01e+3, 1.86e+3,   131.0,   154.0,   139.0,   126.0,   140.0,   131.0,   140.0,   159.0,   120.0,   172.0,   132.0,   118.0,   196.0,   222.0,   116.0,   143.0],
[  308.0,     0.4,   545.0,   497.0,   458.0, 1.74e+3, 1.11e+3,   705.0, 1.42e+3,   549.0,   944.0, 1.71e+3, 1.55e+3,   407.0,   447.0,   421.0,   386.0,   425.0,   407.0,   339.0,   453.0,   350.0,   471.0,   327.0,   355.0,   499.0,   256.0,   367.0,   435.0],
[  240.0,   545.0,       0,    60.9,   105.0, 2.28e+3, 1.65e+3, 1.24e+3, 1.97e+3, 1.09e+3, 1.49e+3, 2.25e+3, 2.09e+3,   165.0,   117.0,   148.0,   194.0,   145.0,   164.0,   263.0,   111.0,   238.0,    88.6,   271.0,   230.0,    58.0,   401.0,   214.0,   130.0],
[  194.0,   497.0,    60.9,       0,    52.1, 2.23e+3,  1.6e+3, 1.19e+3, 1.92e+3, 1.04e+3, 1.44e+3,  2.2e+3, 2.04e+3,   122.0,    77.9,   107.0,   150.0,   103.0,   121.0,   218.0,    72.2,   182.0,    

In [20]:
# Render shortest-paths matrix.
render_matrix(ex_graph.dist)

Matrix([
[1.0e-12,    24.1,   127.0,   118.0,   113.0,    10.5,    22.7,    17.6,    7.39,    2.75,    26.3,    22.3,    16.7,   132.0,   124.0,   117.0,   105.0,   111.0,   124.0,   108.0,    97.0,   106.0,   128.0,   114.0,   122.0,   117.0,   109.0,   125.0,   133.0],
[   24.1, 1.0e-12,   115.0,   105.0,   100.0,    17.5,     2.2,    25.3,    16.7,    21.3,    5.73,    1.77,    10.3,   122.0,   112.0,   107.0,    98.0,    91.5,   116.0,    95.2,    90.6,    92.7,   115.0,   101.0,   109.0,   104.0,    96.4,   112.0,   120.0],
[  127.0,   115.0, 1.0e-12,    99.3,    96.9,   127.0,   113.0,   130.0,   120.0,   125.0,   117.0,   113.0,   111.0,    35.4,     3.1,    39.1,    22.4,    23.2,    23.8,    19.6,    30.4,    90.6,   112.0,    95.5,    99.3,    78.9,    92.9,   109.0,    72.4],
[  118.0,   105.0,    99.3, 1.0e-12,    12.3,   107.0,   102.0,   119.0,   118.0,   115.0,   107.0,   103.0,   110.0,    98.2,    96.2,   114.0,   115.0,   116.0,   110.0,   113.0,   123.0,    11.9,    

In [21]:
# Render error matrix.
render_matrix(out / ex_graph.dist)

Matrix([
[1.41e+11,    12.8,  1.89,  1.65,  1.42,    196.0,    62.4, 57.1, 234.0,    309.0,  47.6,    90.2,   111.0, 0.991,  1.24,  1.19,   1.2,     1.26,    1.05,   1.3,     1.64, 1.13,    1.34,    1.16, 0.965,    1.67, 2.03, 0.925,     1.08],
[    12.8, 4.0e+11,  4.75,  4.75,  4.56,     99.6,   506.0, 27.9,  85.4,     25.7, 165.0,   963.0,   151.0,  3.35,   4.0,  3.92,  3.94,     4.64,     3.5,  3.56,      5.0, 3.78,     4.1,    3.24,  3.26,    4.78, 2.65,  3.27,     3.63],
[    1.89,    4.75,     0, 0.613,  1.08,     18.0,    14.7, 9.55,  16.4,     8.71,  12.7,    19.9,    18.9,  4.66,  37.8,  3.79,  8.65,     6.25,     6.9,  13.4,     3.64, 2.63,   0.792,    2.84,  2.31,   0.735, 4.32,  1.96,     1.79],
[    1.65,    4.75, 0.613,     0,  4.25,     20.9,    15.7, 10.0,  16.3,     9.03,  13.5,    21.4,    18.5,  1.24,  0.81, 0.945,   1.3,    0.888,     1.1,  1.94,    0.585, 15.3,    1.27,    58.1,  8.05,   0.475, 41.9,  7.52,     2.71],
[    1.42,    4.56,  1.08,  4.25,     0,     21

#### Definition of a Custom Loss Function: Graphical Lasso Estimator 

We seek to reproduce the [Graphical Lasso Estimator](https://en.wikipedia.org/wiki/Graphical_lasso) custom loss function within the PyTorch framework. Since such an error and gradient computation function requires a differentiable interpretation of the $L_1$ regularization penalty, we must define a new subclass of `torch.autograd.Function` to implement this regression objective within the working environment.

The Graphical Lasso Estimator is defined through the following mathematical optimizer:

$$\hat{\Theta} = \argmax_{\Theta \succ 0} L(\Theta) = \argmax_{\Theta \succ 0}\left(\log\det(\Theta) - \operatorname{tr}(S\Theta) - \lambda\sum_{i, j}|\Theta_{ij}|\right)$$

Thus, it has the following derivative evaluation:

$$\nabla_{\Theta} L(\Theta) = \frac{1}{\det(\Theta)} \det(\Theta) \Theta^{-\top} - S^T - \lambda \begin{cases}1 & \text{if } \Theta_{ij} > 0 \\ 0 & \text{if } \Theta_{ij} = 0 \\ -1 & \text{if } \Theta_{ij} < 0\end{cases}$$
$$\nabla_{\Theta} L(\Theta) = \Theta^{-1} - S - \lambda \operatorname{sign}(\Theta)$$

In [22]:
LAMBDA = 1e-7


class GraphicalLassoEstimator(torch.autograd.Function):
    @staticmethod
    def forward(ctx, preds, targets):
        """
        Computes the loss value for the Graphical Lasso Estimator loss function.
        """
        ctx.save_for_backward(preds, targets)
        loss = - (preds.det().log() - torch.trace(targets @ preds) - LAMBDA * preds.abs().sum())
        return loss

    @staticmethod
    def backward(ctx, grad_output):
        """
        Computes custom gradients with respect to the inputs. Honors the requirement
        for L1 differentiability within the PyTorch framework.
        """
        preds, targets = ctx.saved_tensors
        grad_predictions = None
        grad_targets = None
        grad = - (preds.inverse() - targets - LAMBDA * preds.sign())
        if ctx.needs_input_grad[0]:
            grad_predictions = grad_output * grad
        if ctx.needs_input_grad[1]:
            grad_targets = grad_output * -grad
        return grad_predictions, grad_targets

#### Fine-Tuning of GNN on Shortest-Path Distances

We preprocess and train the GNN using the steps defined above.

In [23]:
# Add shortest-path data for all nodes of all graphs.
def generate_data_paths(dataset):
    graphs = [utils.scene_graph_dict_to_pyg(v[0][2]) for _, v in dataset.items()]
    for graph in graphs:
        N = graph.num_nodes
        graph.x = graph.x.to(device)
        graph.edge_index = graph.edge_index.to(device)
        graph.paths = torch.zeros((N, N, N)).to(device)
        graph.dist = torch.full((N, N), float('inf')).to(device)
        g = to_networkx(graph, to_undirected=True, edge_attrs=['distance_m'])
        lengths = dict(nx.all_pairs_dijkstra_path_length(g, weight='distance_m'))
        for u in range(N):
            for v in lengths[u]:
                graph.dist[u, v] = lengths[u][v]
                for p in nx.all_shortest_paths(g, u, v, weight='distance_m'):
                    graph.paths[u, v, p] = 1
                    graph.paths[v, u, p] = 1
        graph.dist.fill_diagonal_(EPS)
    return graphs


train_graphs_paths = generate_data_paths(train_dataset)
val_graphs_paths = generate_data_paths(val_dataset)
test_graphs_paths = generate_data_paths(test_dataset)

In [24]:
# Train the GNN to reconstruct the shortest-path distances of the graph.
batch_size = 4
val_freq = 5
epochs = 200
es_patience = 5

def test_loop_paths(dataloader, model, loss_fn, wandb_prefix=None, epoch=None):
    model.to(device)
    model.eval()
    size = len(dataloader.dataset)
    test_loss = error_mod = 0

    with torch.no_grad():
        for graph in dataloader.dataset:
            preds = model(graph)
            test_loss += loss_fn(preds, graph.dist).mean().item()
            error = preds / graph.dist
            error.fill_diagonal_(0)
            error_mod += torch.linalg.matrix_norm(error) / error.shape[0]

    test_loss /= size
    error_mod /= size
    print(f"Test Error: \n Avg error: {error_mod:>0.3f} \n Avg loss: {test_loss:>8f} \n")
    # Log eval metrics under the given split prefix (e.g. 'val') when a run is active.
    if wandb_prefix is not None and wandb.run is not None:
        log = {
            f'{wandb_prefix}/loss': test_loss,
            f'{wandb_prefix}/error_mod': error_mod
        }
        if epoch is not None:
            log['epoch'] = epoch
        wandb.log(log)
        # Final test metrics: also surface as run-summary headline numbers.
        if wandb_prefix == 'test':
            wandb.run.summary.update({k: v for k, v in log.items() if k != 'epoch'})
    return test_loss, error_mod


def train_loop_paths(train_dataloader, val_dataloader, test_dataloader, model, 
               loss_fn, optimizer, scheduler, batch_size=20, epochs=50):
    size = len(train_dataloader.dataset)
    model.to(device)
    model.train()
    val_loss: float = 0
    best_mod, best_state, bad_runs = float('inf'), None, 0
    # Log the full run config: GNN (model_hparams) + optimizer/scheduler/loss/batching.
    run = init_wandb('shortest_path_distances', {
        'batch_size': batch_size, 'epochs': epochs,
        'val_freq': val_freq, 'es_patience': es_patience,
        **optimizer_hparams(optimizer),
        **scheduler_hparams(scheduler),
        **loss_hparams(loss_fn),
    })
    global_step = 0
    for i in range(epochs):
        # Validation loop.
        if i % val_freq == 0:
            print(f"=============\nValidation #{i // val_freq}\n=============")
            val_loss, error_mod = test_loop_paths(val_dataloader, model, loss_fn, wandb_prefix='val', epoch=i)
            if scheduler:
                scheduler.step(val_loss)
            if error_mod < best_mod - 1e-3:
                best_mod, bad_runs = error_mod, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                bad_runs += 1
                if bad_runs >= es_patience:
                    print(f"Early stop at epoch {i} (best error {best_mod:>8f})")
                    break
            model.train()
        
        print(f"=============\nEpoch #{i}\n=============")
        optimizer.zero_grad()
        order = torch.randperm(size)
        for j, idx in enumerate(order.tolist()):
            # Compute prediction and loss.
            graph = train_dataloader.dataset[idx]
            preds = model(graph)
            loss = loss_fn(preds, graph.dist)

            # Backpropagation.
            (loss / graph.num_nodes).backward()

            # Optimization and results.
            if (j + 1) % batch_size == 0:
                clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

                global_step += 1
                loss, current = loss.item(), j
                wandb.log({
                    'train/loss': loss,
                    'train/lr': optimizer.param_groups[0]['lr'],
                    'epoch': i,
                    'global_step': global_step,
                })
                print(f"Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
        
    # Early stopping hatch.
    if best_state is not None:
        model.load_state_dict(best_state)

    # Test the finished model.
    test_loop_paths(test_dataloader, model, loss_fn, wandb_prefix='test')
    run.finish()


# Establish MSE/Graphical-Lasso loss.
loss_fn = nn.MSELoss() # GraphicalLassoEstimator.apply
train_dataloader = DataLoader(train_graphs_paths, batch_size=batch_size)
val_dataloader = DataLoader(val_graphs_paths, batch_size=batch_size)
test_dataloader = DataLoader(test_graphs_paths, batch_size=batch_size)
optimizer = torch.optim.AdamW([
    {'params': spd_gnn.gnn.parameters(), 'lr': 3e-5},
    {'params': spd_gnn.head.parameters(), 'lr': 3e-4},
], lr=3e-4, betas=(0.9, 0.95), weight_decay=0.05)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
train_loop_paths(train_dataloader, val_dataloader, test_dataloader, spd_gnn,
           loss_fn, optimizer, scheduler, batch_size=batch_size, epochs=epochs)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/arushar/.netrc.


wandb: Currently logged in as: arushar (alelab) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


Validation #0
Test Error: 
 Avg error: 36.997 
 Avg loss: 260461.622559 

Epoch #0
Loss: 412461.593750  [    3/   32]
Loss: 198897.765625  [    7/   32]
Loss: 6709.925781  [   11/   32]
Loss: 68457.218750  [   15/   32]
Loss: 3595.548096  [   19/   32]
Loss: 4224.079590  [   23/   32]
Loss: 5619.627930  [   27/   32]
Loss: 40984.281250  [   31/   32]
Epoch #1
Loss: 5171.484375  [    3/   32]
Loss: 11111.529297  [    7/   32]
Loss: 4616.603516  [   11/   32]
Loss: 3885.253906  [   15/   32]
Loss: 27559.605469  [   19/   32]
Loss: 4431.335938  [   23/   32]
Loss: 4815.682617  [   27/   32]
Loss: 2748.034424  [   31/   32]
Epoch #2
Loss: 13421.536133  [    3/   32]
Loss: 3189.467773  [    7/   32]
Loss: 4189.409180  [   11/   32]
Loss: 5790.859375  [   15/   32]
Loss: 4283.017090  [   19/   32]
Loss: 5824.016602  [   23/   32]
Loss: 3257.401367  [   27/   32]
Loss: 5346.562988  [   31/   32]
Epoch #3
Loss: 4320.432129  [    3/   32]
Loss: 3190.739990  [    7/   32]
Loss: 15649.324219  [  

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇████
global_step,▁▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇██
test/error_mod,▁
test/loss,▁
train/loss,█▄▂▇▇▅▅▆▄▇▅▆▇▃▅▃▂▂▃▃▃▃▂▃▃▃▂▃▁▂▂▄▅▁▃▄▃▁▂▃
train/lr,███████████████████▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/error_mod,█▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,125
global_step,1000
test/error_mod,3.98482


In [25]:
torch.save(spd_gnn, '../outputs/e9_multistage_training/spd_gnn_gt.pt')
# spd_gnn = torch.load(f'../outputs/e9_multistage_training/spd_gnn_gt.pt', weights_only=False)

#### Evaluation of Pre-Trained GNN on Edge Incidence and Shortest-Paths Distance Estimation

We test the pre-trained model on the evaluation dataset. We we will render the output error matrix $\mathbf{E}$ for visibility.

In [26]:
# Test out the SPD GNN.
spd_gnn.to(device)
out = spd_gnn(ex_graph)
render_matrix(out)

Matrix([
[    0,    10.5, 130.0, 141.0, 129.0,  68.0,  45.7,   35.6,  55.8,  29.9,  40.5,  69.3,  61.9, 104.0, 111.0,  98.2, 108.0, 103.0, 105.0, 115.0,  100.0, 133.0, 129.0, 142.0, 129.0, 127.0, 150.0,  133.0, 129.0],
[ 10.5, 0.00625, 123.0, 134.0, 123.0,  58.5,  35.9,   25.9,  46.3,  20.6,  30.7,  59.7,  52.2,  99.9, 105.0,  93.7, 104.0,  98.6, 100.0, 112.0,   95.3, 126.0, 122.0, 135.0, 123.0, 120.0, 144.0,  127.0, 123.0],
[130.0,   123.0,     0,  18.9,  23.7, 114.0, 114.0,  114.0, 115.0, 117.0, 114.0, 111.0, 113.0,  59.1,  40.5,  61.0,  65.0,  56.3,  59.4,  77.2,   50.4,  39.7,  19.4,  44.3,  40.0,  17.8,  62.5,   36.1,  25.7],
[141.0,   134.0,  18.9,     0,  18.7, 121.0, 123.0,  124.0, 123.0, 127.0, 124.0, 118.0, 120.0,  70.4,  52.6,  72.3,  75.8,  67.8,  70.8,  87.3,   62.4,  33.1,  15.3,  34.5,  34.4,  16.2,  52.4,   28.8,  20.1],
[129.0,   123.0,  23.7,  18.7,     0, 113.0, 113.0,  113.0, 114.0, 116.0, 114.0, 110.0, 112.0,  57.1,  41.4,  58.3,  62.6,  54.4,  57.9,  74.7,   48.9,

In [27]:
spd_gnn.gate

Parameter containing:
tensor(0.1000, device='cuda:0', requires_grad=True)

In [28]:
# Render shortest-paths matrix.
render_matrix(ex_graph.dist)

Matrix([
[1.0e-12,    24.1,   127.0,   118.0,   113.0,    10.5,    22.7,    17.6,    7.39,    2.75,    26.3,    22.3,    16.7,   132.0,   124.0,   117.0,   105.0,   111.0,   124.0,   108.0,    97.0,   106.0,   128.0,   114.0,   122.0,   117.0,   109.0,   125.0,   133.0],
[   24.1, 1.0e-12,   115.0,   105.0,   100.0,    17.5,     2.2,    25.3,    16.7,    21.3,    5.73,    1.77,    10.3,   122.0,   112.0,   107.0,    98.0,    91.5,   116.0,    95.2,    90.6,    92.7,   115.0,   101.0,   109.0,   104.0,    96.4,   112.0,   120.0],
[  127.0,   115.0, 1.0e-12,    99.3,    96.9,   127.0,   113.0,   130.0,   120.0,   125.0,   117.0,   113.0,   111.0,    35.4,     3.1,    39.1,    22.4,    23.2,    23.8,    19.6,    30.4,    90.6,   112.0,    95.5,    99.3,    78.9,    92.9,   109.0,    72.4],
[  118.0,   105.0,    99.3, 1.0e-12,    12.3,   107.0,   102.0,   119.0,   118.0,   115.0,   107.0,   103.0,   110.0,    98.2,    96.2,   114.0,   115.0,   116.0,   110.0,   113.0,   123.0,    11.9,    

In [29]:
# Render error matrix.
render_matrix(out / ex_graph.dist)

Matrix([
[    0,   0.436,  1.02,   1.2,  1.14,   6.5,  2.01,     2.02,  7.55,  10.8,  1.54,  3.11,    3.71, 0.793, 0.891, 0.843,    1.03,   0.927, 0.845,  1.07,     1.03,  1.26,  1.01,  1.24,  1.06,    1.08,  1.37,     1.07, 0.974],
[0.436, 6.25e+9,  1.07,  1.28,  1.22,  3.35,  16.3,     1.03,  2.78, 0.966,  5.35,  33.7,    5.09, 0.822, 0.945, 0.872,    1.06,    1.08, 0.862,  1.17,     1.05,  1.36,  1.06,  1.34,  1.13,    1.15,   1.5,     1.13,  1.02],
[ 1.02,    1.07,     0,  0.19, 0.245, 0.894,  1.01,    0.877, 0.958, 0.937, 0.977,  0.98,    1.02,  1.67,  13.0,  1.56,     2.9,    2.42,  2.49,  3.95,     1.66, 0.438, 0.174, 0.464, 0.403,   0.226, 0.673,    0.332, 0.355],
[  1.2,    1.28,  0.19,     0,  1.52,  1.13,   1.2,     1.04,  1.05,  1.11,  1.16,  1.15,    1.09, 0.717, 0.547, 0.637,   0.656,   0.583, 0.641, 0.775,    0.505,  2.78,  0.65,  9.29,   1.6,   0.684,  6.39,     1.39, 0.749],
[ 1.14,    1.22, 0.245,  1.52,     0,  1.09,  1.15,    0.987,  1.01,  1.05,  1.11,  1.11,    1.

In [30]:
# Evaluate the GNN on its reconstruction of test graph adjacencies.
detector.gnn.load_state_dict(spd_gnn.gnn.state_dict())
test_dataloader = DataLoader(test_graphs_edges, batch_size=batch_size)
test_loop_edges(test_dataloader, detector, loss_fn)

Test Error: 
 Accuracy: 64.5%, F1: 0.736 | P: 0.586 | R: 0.989 | Bal Acc: 64.5% | Avg loss: 21.489881 



21.48988094329834

In [31]:
# Evaluate the GNN on its reconstruction of test graph shortest path distances.
test_dataloader = DataLoader(test_graphs_paths, batch_size=batch_size)
test_loop_paths(test_dataloader, spd_gnn, loss_fn)

Test Error: 
 Avg error: 3.932 
 Avg loss: 1911.514941 



(1911.51494140625, tensor(3.9316, device='cuda:0'))

### §3 Fine-tuning the GNN to Predict Shortest-Path Subgraph Adjacencies

We now wish to optimize the pre-trained GNN (R-PEARL or Graph Transformer) to classify which nodes reside on the shortest path between two given nodes in the graph. Such a model will serve as the backbone for the multi-stage training loop featured in E9 Multistage Training of the GREP-PRISM project. The equations to represent this procedure are below:
$$\mathbf{H} = \Phi\Big(\mathbb{\hat{E}}_{\mathbf{q}}\big[\Phi(\mathbf{q};\, S, \mathcal{H}\,)\big];\, \mathcal{T}\Big)$$
$$\mathbf{\hat{y}}_{ij} = \text{MLP}\big[\big] \in [0, 1]^N$$

In [32]:
from typing import Union


# Define a class for edge detection and instantiate it.
class GNNShortestPathNavigator(GNNEdgeDetector):
    """
    Simple class to detect whether Node 1 and Node 2 are connected by
    applying an MLP to the Graph Positional Encodings of both nodes concatenated.
    """
    def __init__(self, gnn: Union[RandomGNNPositionalEncodings, GraphTransformer]):
        super().__init__(gnn)
        shape = gnn.out_features
        self.classifier = nn.Sequential(
            nn.Linear(5 * shape, shape),
            nn.LeakyReLU(),
            nn.Linear(shape, 1)
        )

    def forward(self, graph: Data, node1: int, node2: int):
        if self.cached_pe is None or not self.cached_pe.any() or self.graph is not graph:
            self.graph = graph
            self.cached_pe = self.gnn(self.graph)
        pe = self.cached_pe
        hi, hj = pe[node1], pe[node2]
        
        features = torch.cat(
            (pe, hi.expand_as(pe), hj.expand_as(pe), pe * hi, pe * hj), dim=1
        )
        return self.classifier(features)


# Instantiate the class.
navigator = GNNShortestPathNavigator(gnn)

In [33]:
# Test out the Navigator.
navigator.to(device)
node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
out = navigator(ex_graph, node1, node2)
render_matrix(torch.sigmoid(out))

Matrix([
[0.406],
[0.408],
[0.345],
[0.339],
[0.337],
[0.336],
[0.344],
[0.381],
[0.349],
[0.381],
[0.343],
[0.381],
[0.344],
[0.346],
[ 0.34],
[0.359],
[0.353],
[0.359],
[0.357],
[0.349],
[ 0.36],
[0.345],
[ 0.35],
[0.339],
[0.366],
[0.351],
[0.348],
[0.345],
[0.344]])

#### Fine-Tuning of GNN on Shortest-Path Node Inclusion
Finally, we preprocess and train the GNN using the steps defined above.

In [34]:
# Add edge existence tuples for edge existence.
def generate_data_paths(dataset):
    graphs = [utils.scene_graph_dict_to_pyg(v[0][2]) for _, v in dataset.items()]
    for graph in graphs:
        N = graph.num_nodes
        graph.x = graph.x.to(device)
        graph.edge_index = graph.edge_index.to(device)
        graph.paths = torch.zeros((N, N, N)).to(device)
        graph.dist = torch.full((N, N), float('inf')).to(device)
        g = to_networkx(graph, to_undirected=True, edge_attrs=['distance_m'])
        lengths = dict(nx.all_pairs_dijkstra_path_length(g, weight='distance_m'))
        for u in range(N):
            for v in lengths[u]:
                graph.dist[u, v] = lengths[u][v]
                for p in nx.all_shortest_paths(g, u, v, weight='distance_m'):
                    graph.paths[u, v, p] = 1
                    graph.paths[v, u, p] = 1
    
    return graphs

train_graphs_paths = generate_data_paths(train_dataset)
val_graphs_paths = generate_data_paths(val_dataset)
test_graphs_paths = generate_data_paths(test_dataset)

In [35]:
# Train the GNN to reconstruct the eigenvectors of the graph adjacency.
batch_size = 4
val_freq = 5
epochs = 200
es_patience = 5
detour_bce = False

def test_loop_edges(dataloader, model, loss_fn, wandb_prefix=None, epoch=None):
    model.to(device)
    model.eval()
    size = len(dataloader.dataset)
    test_loss = tp = fp = fn = tn = 0

    with torch.no_grad():
        for graph in dataloader.dataset:
            for u in range(graph.num_nodes):
                preds = torch.stack(
                    [model(graph, u, v)for v in range(graph.num_nodes)]
                ).squeeze(-1).to(device)
                test_loss += loss_fn(preds, graph.paths[u]).mean().item() / graph.num_nodes
                true = graph.paths[u].bool()
                pred = preds > 0
                tp += (pred & true).sum().item()
                fp += (pred & ~true).sum().item()
                fn += (~pred & true).sum().item()
                tn += (~pred & ~true).sum().item()

    test_loss /= size
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    bal_acc = 0.5 * (recall + tn / (tn + fp + 1e-9))
    print(f"Test Error: \n F1: {f1:.3f} | P: {precision:.3f} | R: {recall:.3f} | "
          f"Bal Acc: {(100*bal_acc):.1f}% | Avg loss: {test_loss:>8f} \n")
    # Log eval metrics under the given split prefix (e.g. 'val') when a run is active.
    if wandb_prefix is not None and wandb.run is not None:
        log = {
            f'{wandb_prefix}/loss': test_loss,
            f'{wandb_prefix}/f1': f1,
            f'{wandb_prefix}/precision': precision,
            f'{wandb_prefix}/recall': recall,
            f'{wandb_prefix}/bal_acc': bal_acc,
        }
        if epoch is not None:
            log['epoch'] = epoch
        wandb.log(log)
        # Final test metrics: also surface as run-summary headline numbers.
        if wandb_prefix == 'test':
            wandb.run.summary.update({k: v for k, v in log.items() if k != 'epoch'})
    return test_loss, f1


def train_loop_edges(train_dataloader, val_dataloader, test_dataloader, model, 
               loss_fn, optimizer, scheduler, batch_size=20, epochs=50):
    size = len(train_dataloader.dataset)
    model.to(device)
    model.train()
    f1: float = 0
    best_f1, best_state, bad_runs = -float('inf'), None, 0
    # Log the full run config: GNN (model_hparams) + optimizer/scheduler/loss/batching.
    run = init_wandb('path_navigation', {
        'batch_size': batch_size, 'epochs': epochs,
        'val_freq': val_freq, 'es_patience': es_patience,
        'detour_bce': detour_bce,
        **optimizer_hparams(optimizer),
        **scheduler_hparams(scheduler),
        **loss_hparams(loss_fn),
    })
    global_step = 0
    for i in range(epochs):
        # Validation loop.
        if i % val_freq == 0:
            print(f"=============\nValidation #{i // val_freq}\n=============")
            _, f1 = test_loop_edges(val_dataloader, model, loss_fn, wandb_prefix='val', epoch=i)
            if scheduler:
                scheduler.step(f1)
            if f1 > best_f1 + 1e-2:
                best_f1, bad_runs = f1, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                bad_runs += 1
                if bad_runs >= es_patience:
                    print(f"Early stop at epoch {i} (best F1 {best_f1:>8f})")
                    break
            model.train()
        
        print(f"=============\nEpoch #{i}\n=============")
        optimizer.zero_grad()
        for j, graph in enumerate(train_dataloader.dataset):
            # Compute path metrics for Detour-BCE loss.
            N = graph.num_nodes
            if detour_bce:
                D = graph.dist
                diam = D[torch.isfinite(D)].max()
                delta = (D[:, None, :] + D.transpose(0, 1)[None, :, :] - D[:, :, None]).clamp_min(0)
                delta_norm = (delta / diam).nan_to_num(0.0)
                reachable = torch.isfinite(D)

            # Compute prediction and loss.
            loss = 0
            for u in range(N):
                preds = torch.stack(
                    [model(graph, u, v) for v in range(N)]
                ).squeeze(-1).to(device)

                # Compute Detour-BCE loss.
                raw_loss = loss_fn(preds, graph.paths[u])
                if detour_bce:
                    neg_weights = 1.0 + delta_norm[u]
                    weights = torch.where(
                        graph.paths[u].bool(), torch.ones_like(neg_weights), neg_weights
                    )
                    mask = reachable[u].unsqueeze(-1).float()
                loss = loss + raw_loss / graph.num_nodes

            # Backpropagation.
            (loss / batch_size).backward()

            # Optimization and results.
            if (j + 1) % batch_size == 0:
                clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

                global_step += 1
                loss, current = loss.item() / N, j
                wandb.log({
                    'train/loss': loss,
                    'train/lr': optimizer.param_groups[0]['lr'],
                    'epoch': i,
                    'global_step': global_step,
                })
                print(f"Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
        
    # Early stopping hatch.
    if best_state is not None:
        model.load_state_dict(best_state)

    # Test the finished model.
    test_loop_edges(test_dataloader, model, loss_fn, wandb_prefix='test')
    run.finish()


# Establish path-vector (sparsity-) sensitive BCE Logit loss.
positive = sum(g.paths.sum() for g in train_graphs_paths)
pos_weight = (sum(g.paths.numel() for g in train_graphs_paths) - positive) / positive
pos_weight **= 0.5
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device), reduction='mean')

train_dataloader = DataLoader(train_graphs_paths, batch_size=batch_size)
val_dataloader = DataLoader(val_graphs_paths, batch_size=batch_size)
test_dataloader = DataLoader(test_graphs_paths, batch_size=batch_size)

optimizer = torch.optim.AdamW([
    {'params': navigator.gnn.parameters(), 'lr': 3e-5},
    {'params': navigator.classifier.parameters(), 'lr': 3e-4},
], betas=(0.9, 0.95), weight_decay=0.05)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
# train_loop(train_dataloader, val_dataloader, test_dataloader, navigator,
#            loss_fn, optimizer, scheduler, batch_size=batch_size, epochs=epochs)

In [36]:
# torch.save(detector, '../outputs/e9_multistage_training/path_navigator_final_2.pt')
# torch.save(detector.gnn.state_dict(), f'../outputs/e9_multistage_training/path_navigator_{model_type}_final.pt')
navigator = torch.load('../outputs/e9_multistage_training/path_navigator_final.pt', weights_only=False)
gnn.load_state_dict(torch.load(f'../outputs/e9_multistage_training/path_navigator_gt_final.pt'))

<All keys matched successfully>

#### Evaluation of Pre-Trained GNN on Edge Incidence
We thus test the fine-tuned model on the evaluation dataset. First, we we will render the output for clarity.

In [37]:
# Test out the Navigator.
navigator.to(device)
node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
out = navigator(ex_graph, node1, node2)
render_matrix(torch.sigmoid(out))

Matrix([[9.46e-5]])

In [38]:
# Evaluate the GNN on its reconstruction of test graph shortest path distances.
test_loop_edges(test_dataloader, navigator, loss_fn)

ValueError: Target size (torch.Size([33, 33])) must be the same as input size (torch.Size([33]))